<a href="https://colab.research.google.com/github/sruthisripathi/practical-dl/blob/main/finetune_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import socket,warnings
try:
    socket.setdefaulttimeout(1)
    socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(('1.1.1.1', 53))
except socket.error as ex: raise Exception("STOP: No internet. Click '>|' in top right and set 'Internet' switch to on")

In [ ]:
# install libraries

%%capture
# Install the new package (ddgs)
!pip install -U ddgs

In [ ]:
# import libraries

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import requests
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from pathlib import Path
from ddgs import DDGS
from sklearn.model_selection import train_test_split
from torchvision import models, datasets, transforms
from torch.utils.data import Subset, DataLoader
from torch.utils.tensorboard import SummaryWriter

In [ ]:
class TransformFactory:
    def __init__(self, model_name="resnet18"):
        try:
            weights = models.get_model_weights(model_name).DEFAULT
        except Exception:
            weights = models.ResNet18_Weights.DEFAULT
        self.model_name = model_name
        preprocess = weights.transforms()

        raw_size = preprocess.crop_size
        self.img_size = raw_size[0] if isinstance(raw_size, (list, tuple)) else raw_size

        self.mean = preprocess.mean
        self.std = preprocess.std

    def get_model_weights(self):
      return (self.mean, self.std, self.img_size)

    def get_train_transforms(self):
        return transforms.Compose([
            # Now self.img_size is guaranteed to be an int
            transforms.Resize((self.img_size, self.img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=self.mean, std=self.std)
        ])

    def get_eval_transforms(self):
        return transforms.Compose([
            transforms.Resize((self.img_size, self.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=self.mean, std=self.std)
        ])

In [ ]:
class ImageDataLoader:
    def __init__(self, data_path, train_transform, eval_transform, batch_size=32, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
        self.data_path = data_path
        self.batch_size = batch_size

        if not np.isclose(train_ratio + val_ratio + test_ratio, 1.0):
            raise ValueError("Train, validation, and test ratios must sum to 1.")

        base_dataset = datasets.ImageFolder(root=data_path)
        labels = base_dataset.targets

        train_idx, temp_idx = train_test_split(
            np.arange(len(labels)), test_size=(val_ratio + test_ratio), stratify=labels, random_state=42
        )
        val_idx, test_idx = train_test_split(
            temp_idx, test_size=test_ratio/(val_ratio + test_ratio), stratify=[labels[i] for i in temp_idx], random_state=42
        )

        self.train_ds = Subset(datasets.ImageFolder(data_path, train_transform), train_idx)
        self.val_ds = Subset(datasets.ImageFolder(data_path, eval_transform), val_idx)
        self.test_ds = Subset(datasets.ImageFolder(data_path, eval_transform), test_idx)

    def get_loaders(self):
        train_loader = DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, pin_memory=True, num_workers=2)
        val_loader = DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False, pin_memory=True)
        test_loader = DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False, pin_memory=True)
        return train_loader, val_loader, test_loader

In [ ]:
class ImageFetcher:
    def __init__(self, root_dir="image_dataset", max_images=50):
        self.root_dir = root_dir
        self.max_images = max_images
        self.headers = {"User-Agent": "Mozilla/5.0"}

    def fetch_and_save(self, keyword):
        class_path = os.path.join(self.root_dir, "dataset", keyword.replace(" ", "_"))
        os.makedirs(class_path, exist_ok=True)
        print(f"\nFetching images for: {keyword}")

        with DDGS() as ddgs:
            # Get the list of URLs
            imgs = ddgs.images(keyword, max_results=self.max_images)
            results =  [{**item, 'id': i} for i, item in enumerate(imgs, start=1)]

        # Use tqdm to wrap the threaded execution
        with ThreadPoolExecutor(max_workers=10) as executor:
            # We wrap the results in tqdm to see the progress bar
            list(tqdm(executor.map(lambda x: self._download_single(x, class_path), results),
                      total=len(results),
                      desc="Downloading",
                      unit="img"))

    def _download_single(self, item, save_path):
        url = item.get("image")
        try:
            response = requests.get(url, headers=self.headers, timeout=5)
            if response.status_code == 200:
                img = Image.open(BytesIO(response.content)).convert("RGB")
                img.save(os.path.join(save_path, f"{item.get("id")}.jpg"))
        except: pass

In [ ]:
class ModelTrainer:
    def __init__(self, model, device, lr=0.001):
        self.device = device
        self.model = model.to(self.device)
        self.criterion = nn.CrossEntropyLoss()
        trainable_params = [p for p in self.model.parameters() if p.requires_grad]
        self.optimizer = torch.optim.Adam(trainable_params, lr=lr)
        self.writer = SummaryWriter()
        self.best_loss = float('inf')

    def fit_epoch(self, loader):
        self.model.train()
        loss_total = 0
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(self.device), lbls.to(self.device)
            self.optimizer.zero_grad()
            loss = self.criterion(self.model(imgs), lbls)
            loss.backward()
            self.optimizer.step()
            loss_total += loss.item()
        return loss_total / len(loader)

    def evaluate(self, loader):
        self.model.eval()
        v_loss, correct = 0, 0
        with torch.no_grad():
            for imgs, lbls in loader:
                imgs, lbls = imgs.to(self.device), lbls.to(self.device)
                out = self.model(imgs)
                v_loss += self.criterion(out, lbls).item()
                correct += (out.argmax(1) == lbls).sum().item()
        return v_loss / len(loader), 100 * correct / len(loader.dataset)

In [ ]:
class MasterExecutioner:
    def __init__(self, keywords, dataset_path="image_dataset", images_per_chunk=200, iterations=5, batch_size=32, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
        self.keywords = keywords
        self.dataset_path = dataset_path
        self.images_per_chunk = images_per_chunk
        self.iterations = iterations
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.fetcher = ImageFetcher(root_dir=self.dataset_path, max_images=self.images_per_chunk)

        self.batch_size = batch_size
        self.train_ratio = train_ratio
        self.val_ratio = val_ratio
        self.test_ratio = test_ratio
        self.dm = None
        self.t_loader = None
        self.v_loader = None
        self.transformer = None
        self.model = None
        self.trainer = None

    def _init_model(self, model_name):
        model = models.get_model(model_name, weights="DEFAULT")
        for p in model.parameters():
            p.requires_grad = False

        num_classes = len(self.keywords)

        # 2. Identify and replace the correct head attribute
        if hasattr(model, 'fc'):
            model.fc = nn.Linear(model.fc.in_features, num_classes)
        elif hasattr(model, 'classifier'):
            if isinstance(model.classifier, nn.Sequential):
                model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
            else:
                model.classifier = nn.Linear(model.classifier.in_features, num_classes)
        elif hasattr(model, 'heads'):
            # Standard ViT
            model.heads = nn.Linear(model.heads[0].in_features if isinstance(model.heads, nn.Sequential) else model.heads.in_features, num_classes)
        elif hasattr(model, 'head'):
            # Swin Transformer specific fix
            model.head = nn.Linear(model.head.in_features, num_classes)
        else:
            raise AttributeError(f"Architecture {model_name} not supported by current head-replacement logic.")

        self.model = model
        self.trainer = ModelTrainer(self.model, self.device)

    def _init_transformer(self, model_name):
      self.transformer = TransformFactory(model_name)

    def get_model_weights(self, model_name):
        try:
            weights = models.get_model_weights(model_name).DEFAULT
        except Exception:
            weights = models.ResNet18_Weights.DEFAULT

        preprocess = weights.transforms()

        raw_size = preprocess.resize_size
        img_size = raw_size[0] if isinstance(raw_size, (list, tuple)) else raw_size
        mean = preprocess.mean
        std = preprocess.std
        return (mean, std, img_size)

    def should_reload_data(self, model_name):
      if self.transformer:
        return self.transformer.get_model_weights() != self.get_model_weights(model_name)
      return True

    def run_pipeline(self, model_name="resnet18", model_path="best_model.pth", data_fetch=True):
        try:
          data_load = self.should_reload_data(model_name)

          self._init_transformer(model_name)
          self._init_model(model_name)
          if data_fetch:
            for kw in self.keywords: self.fetcher.fetch_and_save(kw)
          if data_load:
            dm = ImageDataLoader(
                self.fetcher.root_dir,
                self.transformer.get_train_transforms(),
                self.transformer.get_eval_transforms(),
                batch_size=self.batch_size,
                train_ratio=self.train_ratio,
                val_ratio=self.val_ratio,
                test_ratio=self.test_ratio
            )
            self.t_loader, self.v_loader, _ = dm.get_loaders()
          for e in range(self.iterations):
              t_loss = self.trainer.fit_epoch(self.t_loader)
              v_loss, v_acc = self.trainer.evaluate(self.v_loader)
              print(f"Epoch {e+1}: Loss {t_loss:.4f} | Val Acc {v_acc:.2f}%")
              if v_loss < self.trainer.best_loss:
                  self.trainer.best_loss = v_loss
                  torch.save(self.model.state_dict(), model_path)
          print(f"\nPipeline complete. Best model saved as {model_path}")
        except KeyboardInterrupt:
            print("\nStopped. Saving current state...")
            torch.save(self.model.state_dict(), model_path)

In [ ]:
class ModelInference:
    def __init__(self, model_name, model_path, classes, factory, device=None):
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.classes = classes
        self.factory = factory
        self.transform = factory.get_eval_transforms()

        # Reconstruct architecture
        self.model = models.get_model(model_name, weights=None)
        self._replace_head(len(classes))

        # Load weights
        self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        self.model.to(self.device).eval()

    def _replace_head(self, num_classes):
        if hasattr(self.model, 'fc'):
            self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)
        elif hasattr(self.model, 'classifier'):
            if isinstance(self.model.classifier, nn.Sequential):
                self.model.classifier[-1] = nn.Linear(self.model.classifier[-1].in_features, num_classes)
            else:
                self.model.classifier = nn.Linear(self.model.classifier.in_features, num_classes)
        elif hasattr(self.model, 'heads'):
            # Standard ViT
            in_f = self.model.heads[0].in_features if isinstance(self.model.heads, nn.Sequential) else self.model.heads.in_features
            self.model.heads = nn.Linear(in_f, num_classes)
        elif hasattr(self.model, 'head'):
            # Swin Transformer specific
            self.model.head = nn.Linear(self.model.head.in_features, num_classes)
        else:
            raise AttributeError(f"Model architecture {type(self.model).__name__} is not supported.")

    def predict(self, image_path):
        img_raw = Image.open(image_path).convert("RGB")
        img_t = self.transform(img_raw).unsqueeze(0).to(self.device)

        with torch.no_grad():
            output = self.model(img_t)
            probs = F.softmax(output, dim=1)
            conf, pred = torch.max(probs, 1)

        label = self.classes[pred.item()]
        score = conf.item() * 100
        print(f"Predicted Label: {label}, Confidence Score: {score}")
        return label, score

    def predict_and_show(self, image_path):
        """Method 1: Single image path with visualization"""
        img_raw = Image.open(image_path).convert("RGB")
        img_t = self.transform(img_raw).unsqueeze(0).to(self.device)

        with torch.no_grad():
            output = self.model(img_t)
            probs = F.softmax(output, dim=1)
            conf, pred = torch.max(probs, 1)

        label = self.classes[pred.item()]
        score = conf.item() * 100

        # --- Visualization Logic ---
        plt.figure(figsize=(6, 6))
        # Convert tensor to numpy and denormalize for display
        img_display = img_t.squeeze(0).cpu().numpy().transpose((1, 2, 0))
        img_display = img_display * self.factory.std + self.factory.mean
        img_display = np.clip(img_display, 0, 1)

        plt.imshow(img_display)
        plt.title(f"Pred: {label} ({score:.2f}%)", fontsize=15, fontweight='bold')
        plt.axis('off')
        plt.show()

        return label, score

    def predict_loader(self, test_loader):
        """Method 2: Full DataLoader for bulk evaluation"""
        all_preds = []
        all_labels = []

        print(f"Running inference on {len(test_loader.dataset)} test images...")
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs = imgs.to(self.device)
                outputs = self.model(imgs)
                _, preds = torch.max(outputs, 1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(lbls.numpy())

        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        acc = (all_preds == all_labels).mean() * 100
        print(f"Test Accuracy: {acc:.2f}%")
        return all_preds, all_labels

In [ ]:
def display_image(image_path_str):
    plt.figure(figsize=(6, 6))
    img = Image.open(image_path_str)
    plt.imshow(img)
    plt.axis('off')
    plt.show()

In [ ]:
m_names = [
    "resnet18",
    "efficientnet_b0",
    "mobilenet_v3_large",
    "vit_b_16",
    "convnext_tiny",
    "convnext_small",
    "swin_t",
    "regnet_y_400mf",
    "efficientnet_v2_s"
]
keywords=["bird", "forest"]

In [ ]:
executioner = MasterExecutioner(
    keywords=keywords,
    images_per_chunk=200,
    iterations=5,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    batch_size=32,
    dataset_path="image_dataset_1",
)
data_fetch = True
# data_fetch = False
for m_name in m_names:
  print("Model: ", m_name)
  executioner.run_pipeline(model_name=m_name, data_fetch=data_fetch, model_path=f"{m_name}.pth")
  data_fetch = False

In [ ]:
im = ImageFetcher("test_dataset", max_images=1)

for kw in keywords:
  im.fetch_and_save(kw)
  image_path = f"{im.root_dir}/dataset/{kw.replace(' ', '_')}/1.jpg"
  print(image_path)
  display_image(image_path)
  for m_name in m_names:
      tramsformer=TransformFactory(m_name)
      infer = ModelInference(model_name=m_name, model_path=f"{m_name}.pth", classes=keywords, factory=tramsformer)
      # preds, targets = infer.predict_loader(tramsformer.get_eval_transforms())
      infer.predict(image_path)